In [23]:
root_dir = ".."
tool_path = "../../detecty-thingy/dist/sql-antipattern-detector"

In [24]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [25]:
# Path to the main tracking dataframe
tracking_file = f"{root_dir}/datasets/final-repositories-tracked.csv"

# Load existing tracking file if it exists, otherwise load original and initialize column
if os.path.exists(tracking_file):
    projects_df = pd.read_csv(tracking_file)
    print("Resuming from existing tracking file.")
else:
    projects_df = pd.read_csv(f"{root_dir}/datasets/final-repositories-corrected.csv")
    projects_df["analysis_runtime"] = None
    print("Starting new analysis tracking.")

Resuming from existing tracking file.


In [26]:
def get_project_dir(project: str):
    return f"{root_dir}/repositories/{project.replace("/", "_")}"

def get_results_file(project: str):
    return f"{root_dir}/datasets/projects/{project.replace("/", "_")}.csv"

In [27]:
import os
import time

for index, row in projects_df.iterrows():
    project = row["name"]
    results_path = get_results_file(project)
    
    # Check if analysis_runtime is already recorded for this project
    if pd.notnull(row.get("analysis_runtime")):
        print(f"Skipping {project}: Already analyzed.")
        continue

    print(f"Analyzing {project}...")
    iter_start_time = time.perf_counter()

    # Run tool and capture output
    cmd = (
        f"{tool_path} {get_project_dir(project)} --format csv "
        f"--model openrouter:anthropic/claude-opus-4.5 "
        f"--temperature 0.0 --thinking-effort none --retries 2"
    )
    
    project_result_df = pd.read_csv(os.popen(cmd))
    
    # Save the specific project results to its own CSV
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    project_result_df.to_csv(results_path, index=False)
    
    iter_end_time = time.perf_counter()
    duration = iter_end_time - iter_start_time
    
    # Update the main tracking dataframe
    projects_df.at[index, "analysis_runtime"] = duration
    
    # Save tracking state immediately so we can resume if interrupted
    projects_df.to_csv(tracking_file, index=False)
    
    print(f"Finished {project} in {duration:.2f} seconds.")

Skipping 4el1k/Spring-Api-Hibernate-JOOQ: Already analyzed.
Skipping 66xc1/lazy-punch: Already analyzed.
Skipping ABCLV/ABLV_IngSW: Already analyzed.
Skipping APSfurizon/fz-backend: Already analyzed.
Skipping AWS-Cloud-Community-LPU/mailing-service: Already analyzed.
Skipping AffectedArc07/ParaUtil: Already analyzed.
Skipping AisleiAvila/empresa: Already analyzed.
Skipping AkaLrz/E-commercial-Platfrom: Already analyzed.
Skipping AlexSpeal/Tracking-Bot: Already analyzed.
Skipping Alexandr-Kokorin/JavaCourse2024: Already analyzed.
Skipping AlexzTj/p2p: Already analyzed.
Skipping AlphaFoxz/oneboot: Already analyzed.
Skipping AnastasiaPleshkova/tnkf-tracker: Already analyzed.
Skipping AndrewSalygin/LinkTracker-Tinkoff: Already analyzed.
Skipping Andrey582/NotificationBot: Already analyzed.
Skipping Ani512/online-tutoring-app-backend: Already analyzed.
Skipping AntonioRusan/LinkTracker: Already analyzed.
Skipping Apress/beginning-jooq: Already analyzed.
Skipping Apress/pro-spring-6: Already

In [28]:
all_results = []
merged_output_path = f"{root_dir}/datasets/analysis-results.csv"

print("Starting merge...")

for _, row in projects_df.iterrows():
    project_name = row["name"]
    file_path = get_results_file(project_name)

    if os.path.exists(file_path):
        try:
            # Read individual project CSV
            df = pd.read_csv(file_path)
            
            # Add the project name as a column so you know where the data came from
            df.insert(0, "project_name", project_name)
            
            all_results.append(df)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"Warning: Expected results file for {project_name} not found at {file_path}")

# Concatenate all dataframes into one
combined_df = pd.concat(all_results, ignore_index=True)

# Save to disk
combined_df.to_csv(merged_output_path, index=False)
print(f"Successfully merged {len(all_results)} projects into: {merged_output_path}")
print(f"Total rows in merged file: {len(combined_df)}")


Starting merge...
Successfully merged 602 projects into: ../datasets/analysis-results.csv
Total rows in merged file: 15931


In [ ]:
Date	Slug	Usage	BYOK Usage	Requests	Prompt Tokens	Completion Tokens	Reasoning Tokens	API Key
2026-04-15 12:00:00	anthropic/claude-opus-4.5	109.559595	0	3785	19087254	564933	0	Unknown
2026-04-15 11:00:00	anthropic/claude-opus-4.5	230.847625	0	5330	42420425	749820	0	Unknown
2026-04-15 10:00:00	anthropic/claude-opus-4.5	145.99265	0	1424	28141785	211349	0	Unknown
2026-04-15 09:00:00	anthropic/claude-opus-4.5	428.51004	0	5057	81312003	878001	0	Unknown
2026-04-15 08:00:00	anthropic/claude-opus-4.5	121.40958	0	2455	22017586	452866	0	Unknown
